# 🔥 Advanced · SAM 2 — segment & track in video

> 🔥 **Advanced / heavy lab.** Click an object on the first frame; SAM 2 tracks its mask through the whole clip.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **10–20 min** including downloads. Built on the official **[facebookresearch/sam2](https://github.com/facebookresearch/sam2)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

Maps to lessons **C5–C6 (hands & hand-object interaction)** — a foundation segmenter for first-person video.

In [ ]:
!nvidia-smi -L

## 1 · Install + checkpoint

In [ ]:
%cd /content
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -q -e .
!mkdir -p checkpoints
!wget -q -O checkpoints/sam2.1_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

## 2 · Extract frames from your egocentric clip
Upload `clip.mp4`; SAM 2's video predictor reads a folder of JPEG frames.

In [ ]:
import os
os.makedirs("frames", exist_ok=True)
!ffmpeg -hide_banner -loglevel error -i clip.mp4 -q:v 2 -start_number 0 frames/%05d.jpg
print("frames:", len(os.listdir("frames")))

## 3 · Click once, then propagate through the video

In [ ]:
import torch, numpy as np
from sam2.build_sam import build_sam2_video_predictor
predictor = build_sam2_video_predictor("configs/sam2.1/sam2.1_hiera_l.yaml", "checkpoints/sam2.1_hiera_large.pt", device="cuda")
state = predictor.init_state(video_path="frames")
# a positive click near the object/hand on frame 0 (edit x,y to your clip):
predictor.add_new_points_or_box(state, frame_idx=0, obj_id=1,
    points=np.array([[320, 240]], dtype=np.float32), labels=np.array([1], np.int32))
video_masks = {}
for frame_idx, obj_ids, mask_logits in predictor.propagate_in_video(state):
    video_masks[frame_idx] = (mask_logits[0] > 0).cpu().numpy()
print("segmented frames:", len(video_masks))

## 4 · Overlay the tracked mask

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
show = sorted(video_masks)[:: max(1, len(video_masks)//4)][:4]
fig, ax = plt.subplots(1, len(show), figsize=(4*len(show), 4))
for a, fi in zip(ax, show):
    a.imshow(mpimg.imread(f"frames/{fi:05d}.jpg")); a.imshow(video_masks[fi].squeeze(), alpha=0.5); a.axis("off"); a.set_title(f"frame {fi}")
plt.show()

## Next steps
- Single images: use `SAM2ImagePredictor`.
- Add multiple `obj_id`s (both hands + the object) and box prompts.
- Feed masks downstream for hand-object interaction analysis (lesson C6).